# DeepLOB "Super Model" (DeepLOB Encoder + Seq2Seq Decoder)

This notebook implements the architecture inspired by the MHF paper.

**Architecture:**
1.  **Encoder:** DeepLOB (Convolutional Blocks + Inception Module + LSTM) to extract rich spatial features from the LOB.
2.  **Decoder:** Seq2Seq LSTM with Additive Attention to generate the 60-second midprice path step-by-step.


In [ ]:
import torch
from torch import nn
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Paths and volume_to_fill
root = Path("/workspace/practical") if Path("/workspace/practical").exists() else Path.cwd()
import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

symbol_dir = root / "BTCUSDT"
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))
print("volume_to_fill:", volume_to_fill)

FEATURE_COLS = [
    "ask_price_1","ask_price_2","ask_price_3","ask_price_4","ask_price_5",
    "bid_price_1","bid_price_2","bid_price_3","bid_price_4","bid_price_5",
    "ask_vol_1","ask_vol_2","ask_vol_3","ask_vol_4","ask_vol_5",
    "bid_vol_1","bid_vol_2","bid_vol_3","bid_vol_4","bid_vol_5",
]

target_window = 60
input_window = 5*60  # 300 seconds input


## 1. The Encoder: DeepLOB (Heavy)
This is the standard DeepLOB CNN-Inception encoder.

In [ ]:
class DeepLOBEncoder(nn.Module):
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=(1,2), stride=(1,1)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32,32,kernel_size=(1,2),stride=(1,1)), nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.Tanh(), nn.BatchNorm2d(32),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32,32,kernel_size=(1,2)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32,32,kernel_size=(3,1), padding=(1,0)), nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.inp1 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64,64,kernel_size=(3,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp2 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64,64,kernel_size=(5,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp3 = nn.Sequential(
            nn.MaxPool2d((3,1),stride=(1,1),padding=(1,0)),
            nn.Conv2d(32,64,kernel_size=(1,1),padding='same'), nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
    def forward(self, x):
        # x: [B, T, F]
        B, T, F = x.shape
        x = x.view(B*T, 1, 5, 4)
        x = self.conv1(x); x = self.conv2(x); x = self.conv3(x)
        x = torch.cat([self.inp1(x), self.inp2(x), self.inp3(x)], dim=1)
        x = x.permute(0,2,1,3).reshape(x.size(0), x.size(2), -1)
        x = x.mean(dim=1)
        return x.view(B, T, -1) # [B, T, 192]


## 2. The Decoder: Seq2Seq Attention
This replaces the simple Bi-LSTM head with a full autoregressive decoder.

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, enc_dim: int, dec_dim: int, attn_dim: int = 64):
        super().__init__()
        self.W_e = nn.Linear(enc_dim, attn_dim, bias=False)
        self.W_d = nn.Linear(dec_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, enc_outputs: torch.Tensor, dec_state: torch.Tensor) -> torch.Tensor:
        # enc_outputs: [B, T, E], dec_state: [B, D]
        e = self.W_e(enc_outputs) + self.W_d(dec_state).unsqueeze(1)  # [B, T, A]
        scores = self.v(torch.tanh(e)).squeeze(-1)  # [B, T]
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(weights.unsqueeze(-1) * enc_outputs, dim=1)  # [B, E]
        return context

class SuperModel(nn.Module):
    def __init__(self, hidden: int = 128, horizon: int = 60, dropout: float=0.1):
        super().__init__()
        # 1. Encoder Part
        self.spatial = DeepLOBEncoder(in_ch=1) # The Heavy CNN [B, T, 192]
        self.encoder = nn.LSTM(
            input_size=192, # DeepLOB output dim
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # 2. Decoder Part
        self.attn = AdditiveAttention(enc_dim=2 * hidden, dec_dim=hidden)
        self.decoder = nn.LSTM(
            input_size=1 + 2 * hidden,  # previous y + context
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            dropout=0.0,
        )
        self.out = nn.Linear(hidden, 1)
        self.init_h = nn.Linear(2 * hidden, hidden)
        self.init_c = nn.Linear(2 * hidden, hidden)
        self.horizon = horizon

    def forward(self, x: torch.Tensor, y_teacher: torch.Tensor = None) -> torch.Tensor:
        # x: [B, T, F]
        # y_teacher: [B, 60]
        
        # Encode
        h_spatial = self.spatial(x)  # [B, T, 192]
        enc_out, _ = self.encoder(h_spatial)  # [B, T, 2H]

        # Init decoder state from encoder mean
        enc_mean = enc_out.mean(dim=1)
        dec_h0 = torch.tanh(self.init_h(enc_mean))
        dec_c0 = torch.tanh(self.init_c(enc_mean))
        dec_h = dec_h0.unsqueeze(0)  # [1, B, H]
        dec_c = dec_c0.unsqueeze(0)

        outputs = []
        # Seed: use last encoder input midprice proxy (col 0 normalized)
        prev_y = x[:, -1, 0].unsqueeze(-1) # [B, 1]

        for t in range(self.horizon):
            context = self.attn(enc_out, dec_h[-1])  # [B, 2H]
            dec_in = torch.cat([prev_y, context], dim=1).unsqueeze(1)  # [B, 1, 1+2H]
            dec_out, (dec_h, dec_c) = self.decoder(dec_in, (dec_h, dec_c))
            y_hat = self.out(dec_out).squeeze(1)  # [B, 1]
            outputs.append(y_hat.squeeze(-1))
            if self.training and y_teacher is not None:
                # Teacher forcing
                prev_y = y_teacher[:, t].unsqueeze(-1)
            else:
                # Autoregressive
                prev_y = y_hat.detach()

        return torch.stack(outputs, dim=1)  # [B, 60]

def forecast_loss(pred, target, smooth_lambda=0.02):
    mse = (pred - target) ** 2
    loss = mse.mean()
    if smooth_lambda > 0:
        smooth = (pred[:, 1:] - pred[:, :-1]) ** 2
        loss = loss + smooth_lambda * smooth.mean()
    return loss


## 3. Dataset & Training Loop

In [ ]:
# Normalized dataset
class LastMinuteDatasetNorm(Dataset):
    def __init__(self, x_df: pd.DataFrame, y_df: pd.DataFrame, feat_means, feat_stds, mid_mean, mid_std):
        self.samples = []
        for hid, xh in x_df.groupby("anonymized_id"):
            yh = y_df[y_df["anonymized_id"] == hid]
            if len(yh) != target_window:
                continue
            xh = xh.sort_values("time_in_hour").tail(input_window)
            x_arr = xh[FEATURE_COLS].astype(np.float32).to_numpy()
            if x_arr.shape[0] < input_window:
                pad = np.zeros((input_window - x_arr.shape[0], x_arr.shape[1]), dtype=np.float32)
                x_arr = np.vstack([pad, x_arr])
            mid = ((yh["ask_price_1"] + yh["bid_price_1"]) / 2.0).astype(np.float32).to_numpy()
            if mid.shape[0] != target_window:
                continue
            x_arr = (x_arr - feat_means) / feat_stds
            mid_norm = (mid - mid_mean) / mid_std
            if not np.isfinite(x_arr).all() or not np.isfinite(mid_norm).all():
                continue
            self.samples.append((x_arr, mid_norm))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.from_numpy(y)

from dataclasses import dataclass

def chrono_split(x_df, y_df, val_ratio=0.2):
    ids = np.sort(x_df["anonymized_id"].unique())
    split = int(len(ids)*(1-val_ratio))
    tr_ids, va_ids = ids[:split], ids[split:]
    x_tr = x_df[x_df["anonymized_id"].isin(tr_ids)].reset_index(drop=True)
    x_va = x_df[x_df["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    y_tr = y_df[y_df["anonymized_id"].isin(tr_ids)].reset_index(drop=True)
    y_va = y_df[y_df["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    return x_tr, x_va, y_tr, y_va

@dataclass
class TrainCfg:
    epochs: int = 15
    batch_size: int = 16
    lr: float = 1e-3
    weight_decay: float = 1e-5
    smooth_lambda: float = 0.02
    val_ratio: float = 0.2

def train_val():
    X = pd.read_parquet(X_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    Y = pd.read_parquet(Y_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    x_tr, x_va, y_tr, y_va = chrono_split(X, Y, val_ratio=TrainCfg.val_ratio)

    feat_means = x_tr[FEATURE_COLS].mean().to_numpy().astype(np.float32)
    feat_stds  = x_tr[FEATURE_COLS].std().replace(0,1e-6).to_numpy().astype(np.float32)
    mid_tr = (y_tr["ask_price_1"] + y_tr["bid_price_1"]) / 2.0
    mid_mean = float(mid_tr.mean()); mid_std = float(mid_tr.std() if mid_tr.std()!=0 else 1e-6)

    train_ds = LastMinuteDatasetNorm(x_tr, y_tr, feat_means, feat_stds, mid_mean, mid_std)
    val_ds   = LastMinuteDatasetNorm(x_va, y_va, feat_means, feat_stds, mid_mean, mid_std)
    train_loader = DataLoader(train_ds, batch_size=TrainCfg.batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=TrainCfg.batch_size, shuffle=False)

    model = SuperModel().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=TrainCfg.lr, weight_decay=TrainCfg.weight_decay)

    for epoch in range(TrainCfg.epochs):
        tr = train_epoch(model, train_loader, opt, TrainCfg)
        va = eval_epoch(model, val_loader, TrainCfg)
        print(f"epoch {epoch+1}: train={tr:.4f} val={va:.4f}")

    scalers = {"feat_means": feat_means, "feat_stds": feat_stds, "mid_mean": mid_mean, "mid_std": mid_std}
    return model, scalers, val_loader

def train_epoch(model, loader, opt, cfg: TrainCfg):
    model.train(); total=0; n=0
    for xb, yb in loader:
        xb = xb.to(device); yb = yb.to(device)
        opt.zero_grad()
        # Pass yb as teacher
        pred = model(xb, y_teacher=yb)
        loss = forecast_loss(pred, yb, cfg.smooth_lambda)
        loss.backward(); opt.step()
        total += float(loss.item())*xb.size(0); n += xb.size(0)
    return total/max(n,1)

def eval_epoch(model, loader, cfg: TrainCfg):
    model.eval(); total=0; n=0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            # No teacher forcing during eval
            pred = model(xb, y_teacher=None)
            loss = forecast_loss(pred, yb, cfg.smooth_lambda)
            total += float(loss.item())*xb.size(0); n += xb.size(0)
    return total/max(n,1)


In [ ]:
# Train
run_train = True
if run_train and X_path.exists() and Y_path.exists():
    model, scalers, val_loader = train_val()
else:
    print("Set run_train=True and ensure X_path/Y_path exist.")

## 4. Evaluation
Calculate implementation error (bps) usage `alpha=1.0` (pure model).

In [ ]:
from simulate_walk_the_book import simulate_walk_the_book
try:
    from predictive_scheduler import schedule_from_forecasts
except ImportError:
    from predictive_scheduler import build_schedule_from_forecasts as schedule_from_forecasts

run_eval = True
if run_eval and X_path.exists() and Y_path.exists() and 'model' in globals():
    X = pd.read_parquet(X_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    Y = pd.read_parquet(Y_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    ids = np.sort(X["anonymized_id"].unique()); split = int(len(ids)*0.8); va_ids = ids[split:]
    x_va_raw = X[X["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    y_va_raw = Y[Y["anonymized_id"].isin(va_ids)].reset_index(drop=True)

    errs = []
    model.eval()
    for hid in va_ids:
        xh_raw = x_va_raw[x_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
        yh_raw = y_va_raw[y_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
        if len(yh_raw)!=target_window:
            continue
        # normalize input
        feat_means = scalers['feat_means']; feat_stds = scalers['feat_stds']; mid_mean = scalers['mid_mean']; mid_std = scalers['mid_std']
        xh = xh_raw.tail(input_window)
        # Pad if needed
        x_arr = xh[FEATURE_COLS].astype(np.float32).to_numpy()
        if x_arr.shape[0] < input_window:
            pad = np.zeros((input_window - x_arr.shape[0], x_arr.shape[1]), dtype=np.float32)
            x_arr = np.vstack([pad, x_arr])
        x_arr = (x_arr - feat_means) / feat_stds
        xb = torch.from_numpy(x_arr).unsqueeze(0).to(device)
        with torch.no_grad():
            # No teacher forcing during eval
            pred = model(xb, y_teacher=None)
        mid_pred = (pred.cpu().numpy()[0] * mid_std) + mid_mean

        # Build positions with alpha=1.0
        def end_window_twap(vol, window=14):
            w = np.zeros(60, dtype=np.float32); w[-window:] = vol / window; return w
        sched_forecast = schedule_from_forecasts(mid_pred, volume_to_fill)
        sched_twap = end_window_twap(volume_to_fill, window=14)
        alpha = 1.0  # FULL MODEL CONFIDENCE
        positions = alpha * sched_forecast + (1-alpha) * sched_twap
        positions = positions * (volume_to_fill / positions.sum()) if positions.sum()!=0 else sched_twap

        ask_prices = np.column_stack([yh_raw[f"ask_price_{lvl}"].to_numpy(dtype=float) for lvl in range(1,6)])
        ask_vols   = np.column_stack([yh_raw[f"ask_vol_{lvl}"].to_numpy(dtype=float)   for lvl in range(1,6)])
        bid_prices = np.column_stack([yh_raw[f"bid_price_{lvl}"].to_numpy(dtype=float) for lvl in range(1,6)])
        bid_vols   = np.column_stack([yh_raw[f"bid_vol_{lvl}"].to_numpy(dtype=float)   for lvl in range(1,6)])
        total_vol, avg_price = simulate_walk_the_book(positions, ask_prices, ask_vols, bid_prices, bid_vols)
        close_price = yh_raw["close"].dropna().iloc[-1] if len(yh_raw["close"].dropna())>0 else np.nan
        if total_vol>0 and not np.isnan(avg_price) and not np.isnan(close_price):
            rel_err = abs(avg_price - close_price) / close_price
            fill_penalty = min(100.0, volume_to_fill / total_vol)
            err_bps = rel_err * fill_penalty * 10000.0
            errs.append(err_bps)
    if errs:
        print(f"Validation implementation error (bps) over {len(errs)} hours: mean={np.mean(errs):.4f}, median={np.median(errs):.4f}")
    else:
        print("No valid errors computed.")
else:
    print("Set run_eval=True after training to evaluate.")

In [ ]:
# R^2 Score Calculation
run_r2 = True
if run_r2 and X_path.exists() and Y_path.exists() and 'model' in globals():
    from sklearn.metrics import r2_score
    
    X = pd.read_parquet(X_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    Y = pd.read_parquet(Y_path).sort_values(["anonymized_id","time_in_hour"]).reset_index(drop=True)
    ids = np.sort(X["anonymized_id"].unique()); split = int(len(ids)*0.8); va_ids = ids[split:]
    x_va_raw = X[X["anonymized_id"].isin(va_ids)].reset_index(drop=True)
    y_va_raw = Y[Y["anonymized_id"].isin(va_ids)].reset_index(drop=True)

    all_preds = []
    all_targets = []

    model.eval()
    with torch.no_grad():
        for hid in va_ids:
            xh_raw = x_va_raw[x_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
            yh_raw = y_va_raw[y_va_raw["anonymized_id"]==hid].sort_values("time_in_hour")
            if len(yh_raw)!=target_window:
                continue
            
            # Get Ground Truth Midprice
            mid_true = ((yh_raw["ask_price_1"] + yh_raw["bid_price_1"]) / 2.0).to_numpy()

            # Prepare Input
            feat_means = scalers['feat_means']; feat_stds = scalers['feat_stds']; mid_mean = scalers['mid_mean']; mid_std = scalers['mid_std']
            xh = xh_raw.tail(input_window)
            x_arr = xh[FEATURE_COLS].astype(np.float32).to_numpy()
            if x_arr.shape[0] < input_window:
                pad = np.zeros((input_window - x_arr.shape[0], x_arr.shape[1]), dtype=np.float32)
                x_arr = np.vstack([pad, x_arr])
            x_arr = (x_arr - feat_means) / feat_stds
            xb = torch.from_numpy(x_arr).unsqueeze(0).to(device)
            
            # Predict
            # No teacher forcing during eval
            pred_norm = model(xb, y_teacher=None)
            mid_pred = (pred_norm.cpu().numpy()[0] * mid_std) + mid_mean
            
            all_preds.append(mid_pred)
            all_targets.append(mid_true)
    
    if all_preds:
        # Flatten arrays: we care about R2 over all predicted seconds
        flat_preds = np.concatenate(all_preds)
        flat_targets = np.concatenate(all_targets)
        
        # MASK NANS to avoid ValueError
        mask = np.isfinite(flat_preds) & np.isfinite(flat_targets)
        valid_preds = flat_preds[mask]
        valid_targets = flat_targets[mask]
        
        if len(valid_preds) > 0:
            r2 = r2_score(valid_targets, valid_preds)
            print(f"Validation R^2 Score (all 60s steps): {r2:.4f}")
            print(f"(Evaluated on {len(valid_preds)} valid points, {len(flat_preds)-len(valid_preds)} NaNs removed)")
        else:
            print("All predictions/targets were NaN or missing.")

    else:
        print("No predictions made.")
else:
    print("Set run_r2=True to evaluate.")